# ISRO Hackathon — Surface AQI (Objective 1): Physics-Informed ConvLSTM Pipeline

**Run order:** Execute cells top-to-bottom exactly once, starting with **Section 0** (Colab setup).

**Scope:** Objective 1 only — PM2.5, PM10, CO via ConvLSTM; SO2, NO2 via CNN+XGBoost (lag-free).
HCHO and Objective 2 are excluded.

**Architecture:** Split-model — ConvLSTM (transport-dominated) + CNN+XGBoost (emission-dominated)
+ O3 sub-model + BMA weighting hook.

**Known limitations (stated upfront):**
- AOD monsoon gaps (Jun–Sep ~88% NaN) — masked, not interpolated.
- Pressure-level RH not available (ERA5 pull had only temperature `t`); lapse rate is temperature-only.
- Advection-diffusion PDE uses constant K, neglects emission source S.
- 2020 COVID lockdown flagged (not excluded) via `lockdown_flag` channel.
- XGBoost stubs (BMA layer, O3 sub-model) require real XGBoost outputs to be trained end-to-end.


## 0. Colab Runtime Setup

**Run this first.** Select GPU: Runtime → Change runtime type → T4 GPU.

In [ ]:
# Mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(["pip", "install", "-q", "optuna", "xgboost", "tensorflow"], check=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Imports

In [ ]:
import os
import re
import json
import random
from pathlib import Path
from datetime import date, timedelta

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

try:
    import optuna
except ImportError:
    optuna = None
    print("optuna not installed -- hyperparameter search will be unavailable.")

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


## 2. Global Configuration

Edit `DRIVE_ROOT` if your Google Drive folder layout differs.
Everything else auto-detects from the actual data files.


In [ ]:
# ── Grid (canonical full-res -- auto-overridden below if downsampled data is present)
GRID_HEIGHT = 640
GRID_WIDTH  = 591
TARGET_BOUNDS = (67.9503108224518, 6.450029504117943, 97.50044599248051, 38.450175881137184)
TARGET_CRS = "EPSG:4326"

# ── Date range
START_DATE     = date(2020, 1, 1)
END_DATE       = date(2023, 12, 31)
LOCKDOWN_START = date(2020, 3, 25)   # COVID lockdown -- flagged, not excluded
LOCKDOWN_END   = date(2020, 5, 31)

# ── Pollutant assignments
CONVLSTM_POLLUTANTS = ["PM2.5", "PM10", "CO"]   # this notebook's ConvLSTM branch
XGBOOST_POLLUTANTS  = ["SO2", "NO2"]             # CNN+XGBoost branch (lag-free)
O3_INPUT_POLLUTANTS = ["NO2"]                    # NOTE: HCHO removed -- NO2-only O3 sub-model

# ── Sequence
LOOKBACK_DAYS    = 14
PREDICT_HORIZON  = 1

# ── Paths (Google Drive)
DRIVE_ROOT          = Path("/content/drive/MyDrive/objective_1")
FINAL_INPUT_DIR     = DRIVE_ROOT / "final_input"       # input_YYYY-MM-DD.npy  (18, 640, 591) float32
CHANNEL_METADATA_PATH = DRIVE_ROOT / "channel_metadata.json"
TRAIN_STATIONS_CSV  = DRIVE_ROOT / "train_stations_final.csv"
VAL_STATIONS_CSV    = DRIVE_ROOT / "val_stations_final.csv"
CPCB_OVERPASS_DIR   = DRIVE_ROOT / "cpcb_overpass"         # 268 training-station CSVs
CPCB_OVERPASS_VAL_DIR = CPCB_OVERPASS_DIR / "val"          # 66 validation-station CSVs

# ── Physics-loss weights (small -- physics as regularizer, not hard constraint)
LAMBDA_PM_ORDERING  = 0.05
LAMBDA_PDE_RESIDUAL = 0.02
LAMBDA_PEARSON      = 0.10
DIFFUSIVITY_K       = 50.0   # m²/s constant (simplified)
EMISSION_SOURCE_S   = 0.0    # neglected (no emissions inventory in dataset)

# ── Reproducibility
GLOBAL_SEEDS = [13, 42, 77]

# ── Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load channel metadata written by Stage 2 merge script
with open(CHANNEL_METADATA_PATH) as _f:
    CHANNEL_METADATA = json.load(_f)
INPUT_CHANNEL_NAMES = CHANNEL_METADATA["input_channels"]  # 18 names in order

# ── Auto-detect grid resolution from an actual input file
# Supports full-res .npy (640x591) and downsampled .npz (e.g. 160x148)
def _find_sample_input(d: Path):
    for ext in (".npz", ".npy"):
        matches = sorted(d.glob(f"input_*{ext}"))
        if matches:
            return matches[0], ext
    raise FileNotFoundError(f"No input_*.npy or input_*.npz found in {d}")

_sample_path, INPUT_FILE_EXT = _find_sample_input(FINAL_INPUT_DIR)
_arr = np.load(_sample_path)["data"] if INPUT_FILE_EXT == ".npz" else np.load(_sample_path)
assert _arr.shape[0] == len(INPUT_CHANNEL_NAMES), (
    f"Channel count mismatch: file has {_arr.shape[0]}, "
    f"channel_metadata.json lists {len(INPUT_CHANNEL_NAMES)}"
)
GRID_HEIGHT, GRID_WIDTH = _arr.shape[1], _arr.shape[2]
del _arr

if (GRID_HEIGHT, GRID_WIDTH) != tuple(CHANNEL_METADATA["grid_shape"]):
    print(f"NOTE: using downsampled grid {(GRID_HEIGHT, GRID_WIDTH)} "
          f"(full-res is {tuple(CHANNEL_METADATA['grid_shape'])}). "
          f"All downstream functions scale automatically.")

print(f"Config ready. Device: {DEVICE}")
print(f"Grid: {GRID_HEIGHT}x{GRID_WIDTH} | Channels ({len(INPUT_CHANNEL_NAMES)}): {INPUT_CHANNEL_NAMES}")


## 3. FAST_CONFIG — T4 Time Budget (target: 6-7 hours)

| Component | Time |
|---|---|
| ConvLSTM training (1 seed, 20 epochs) | ~45-60 min |
| Optuna search (6 trials × 8 epochs) | ~1.5-2 hrs |
| CNN+XGBoost branch (SO2, NO2) | ~45-60 min |
| Stratified LOSO (12 stations, 10 epochs/fold) | ~2.5-3.5 hrs |
| **Total** | **~5.5-7 hrs** |

Set `FAST_CONFIG = False` to restore full-budget settings.


In [ ]:
FAST_CONFIG = True   # <-- flip to False for full-budget run

if FAST_CONFIG:
    DOWNSAMPLE_FACTOR   = 4          # grid training resolution: 160x148
    FAST_GLOBAL_SEEDS   = [42]
    FAST_N_EPOCHS       = 20
    FAST_OPTUNA_TRIALS  = 6
    FAST_OPTUNA_EPOCHS  = 8
    FAST_LOSO_N_STATIONS = 12
    FAST_LOSO_EPOCHS    = 10
    FAST_BATCH_SIZE     = 8
    print(f"FAST_CONFIG ON: grid ~{GRID_HEIGHT//4}x{GRID_WIDTH//4}, "
          f"1 seed, {FAST_N_EPOCHS} epochs, {FAST_OPTUNA_TRIALS} Optuna trials, "
          f"LOSO on {FAST_LOSO_N_STATIONS} stations.")
else:
    DOWNSAMPLE_FACTOR   = 1
    FAST_GLOBAL_SEEDS   = GLOBAL_SEEDS
    FAST_N_EPOCHS       = 50
    FAST_OPTUNA_TRIALS  = 20
    FAST_OPTUNA_EPOCHS  = 15
    FAST_LOSO_N_STATIONS = None
    FAST_LOSO_EPOCHS    = 50
    FAST_BATCH_SIZE     = 4
    print("FAST_CONFIG OFF — expect multi-day training on T4.")


## 4. Data Loading

Loads the Stage-2-merged daily `.npy` stacks (18 channels, already aligned onto the 0.05°
TROPOMI grid by the merge script).  RH and lapse_rate are already pre-computed channels —
they do NOT need to be recomputed here.


In [ ]:
def load_daily_stack(date_str: str) -> dict:
    """
    Load input_YYYY-MM-DD.npy (or .npz) and return {channel_name: 2D array}.
    All 18 channels including TROPOMI gases, ERA5 met, derived RH/lapse_rate,
    static lat/lon_norm, and lockdown_flag are present in this file.
    """
    path = FINAL_INPUT_DIR / f"input_{date_str}{INPUT_FILE_EXT}"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Ensure FINAL_INPUT_DIR points at the "
            "Stage-2 final_input/ folder in your Drive."
        )
    arr = np.load(path)["data"] if INPUT_FILE_EXT == ".npz" else np.load(path)
    return {name: arr[i] for i, name in enumerate(INPUT_CHANNEL_NAMES)}


# ── Reference functions kept for documentation / reuse outside main pipeline ──

def relative_humidity_from_t_td(t2m_K: np.ndarray, d2m_K: np.ndarray) -> np.ndarray:
    """Magnus formula: RH(%) from t2m and d2m (both in Kelvin).
    NOT called by load_daily_stack — rh_surface is already baked into the .npy stack."""
    t_c  = t2m_K - 273.15
    td_c = d2m_K - 273.15
    def svp(tc): return 6.112 * np.exp((17.67 * tc) / (tc + 243.5))
    return np.clip(100.0 * svp(td_c) / svp(t_c), 0, 100)


def lapse_rate_850_1000(t_pl: np.ndarray, levels: list) -> np.ndarray:
    """T(850hPa) - T(1000hPa) inversion proxy (temperature-only — see Known Limitations).
    NOT called by load_daily_stack — lapse_rate is already baked into the .npy stack."""
    return t_pl[levels.index(850.0)] - t_pl[levels.index(1000.0)]


## 5. CPCB Station Data Loading

In [ ]:
def _safe_station_name(station_name: str) -> str:
    """Matches the naming convention used by extract_cpcb_overpass.py:
    lowercase, non-alphanumeric runs collapsed to single underscore."""
    s = station_name.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return re.sub(r"_+", "_", s).strip("_")


def load_cpcb_overpass(station_name: str, cpcb_dir: Path = CPCB_OVERPASS_DIR) -> pd.DataFrame:
    """Load pre-extracted overpass CSV for one station (one row per day).
    Pass cpcb_dir=CPCB_OVERPASS_VAL_DIR for held-out validation stations."""
    safe = _safe_station_name(station_name)
    path = cpcb_dir / f"{safe}_overpass.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")
    return pd.read_csv(path, parse_dates=["date"])


def extract_overpass_hour_target(overpass_df: pd.DataFrame, date_str: str) -> dict:
    """Return the pollutant values for a specific date from an overpass CSV."""
    target_date = pd.Timestamp(date_str).date()
    row = overpass_df[overpass_df["date"].dt.date == target_date]
    return row.iloc[0].to_dict() if not row.empty else {}


## 6. Station-to-Pixel Mapping & Sparse Label Grid

In [ ]:
def latlon_to_pixel(lat: float, lon: float) -> tuple:
    """Convert lat/lon to (row, col) on the active grid (scales with GRID_HEIGHT/WIDTH)."""
    left, bottom, right, top = TARGET_BOUNDS
    col = int((lon - left)  / (right - left) * GRID_WIDTH)
    row = int((top  - lat)  / (top - bottom) * GRID_HEIGHT)
    return int(np.clip(row, 0, GRID_HEIGHT - 1)), int(np.clip(col, 0, GRID_WIDTH - 1))


def build_station_pixel_lookup(stations_df: pd.DataFrame) -> dict:
    return {r["station_name"]: latlon_to_pixel(r["latitude"], r["longitude"])
            for _, r in stations_df.iterrows()}


# Load station lists once -- reused throughout
train_stations_df   = pd.read_csv(TRAIN_STATIONS_CSV)
val_stations_df     = pd.read_csv(VAL_STATIONS_CSV)
TRAIN_PIXEL_LOOKUP  = build_station_pixel_lookup(train_stations_df)
VAL_PIXEL_LOOKUP    = build_station_pixel_lookup(val_stations_df)


def populate_target_grid(date_str: str, pollutants: list,
                          use_val_stations: bool = False) -> np.ndarray:
    """
    Build [n_pollutants, GRID_HEIGHT, GRID_WIDTH] with NaN everywhere except
    at CPCB station pixels that have a real overpass-hour reading for this date.
    NaN pixels are EXCLUDED from the loss (not treated as zero-error).
    use_val_stations=True uses the held-out val stations (never seen during training).
    """
    stations_df = val_stations_df if use_val_stations else train_stations_df
    lookup      = VAL_PIXEL_LOOKUP if use_val_stations else TRAIN_PIXEL_LOOKUP
    cpcb_dir    = CPCB_OVERPASS_VAL_DIR if use_val_stations else CPCB_OVERPASS_DIR

    grid = np.full((len(pollutants), GRID_HEIGHT, GRID_WIDTH), np.nan, dtype="float32")

    for _, row in stations_df.iterrows():
        try:
            df = load_cpcb_overpass(row["station_name"], cpcb_dir)
        except FileNotFoundError:
            continue
        vals = extract_overpass_hour_target(df, date_str)
        r, c = lookup[row["station_name"]]
        for i, pol in enumerate(pollutants):
            if pol in vals and pd.notna(vals[pol]):
                grid[i, r, c] = vals[pol]
    return grid


## 7. Adaptive Neighbor Selection (KSC-ConvLSTM inspired)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = (np.sin((lat2-lat1)/2)**2 +
         np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2)
    return 2 * R * np.arcsin(np.sqrt(a))


def select_adaptive_neighbors(target_station: dict, candidate_stations: pd.DataFrame,
                               station_timeseries: dict, k: int = 5,
                               max_distance_km: float = 150.0,
                               min_correlation: float = 0.4) -> list:
    """KSC-style: distance filter + correlation threshold (not distance alone)."""
    cands = candidate_stations.copy()
    cands["distance_km"] = haversine_km(
        target_station["latitude"], target_station["longitude"],
        cands["latitude"].values, cands["longitude"].values)
    cands = cands[(cands["distance_km"] <= max_distance_km) &
                  (cands["station_name"] != target_station["station_name"])]

    if target_station["station_name"] not in station_timeseries:
        return cands.nsmallest(k, "distance_km")["station_name"].tolist()

    tgt = station_timeseries[target_station["station_name"]]
    keep = []
    for _, row in cands.iterrows():
        cand = station_timeseries.get(row["station_name"])
        if cand is None:
            continue
        aligned = pd.concat([tgt, cand], axis=1, join="inner").dropna()
        if len(aligned) >= 30 and aligned.iloc[:,0].corr(aligned.iloc[:,1]) >= min_correlation:
            keep.append(row["station_name"])
    return cands[cands["station_name"].isin(keep)].nsmallest(k, "distance_km")["station_name"].tolist()


## 8. Sequence Construction (ConvLSTM input)

In [ ]:
class AQISequenceDataset(Dataset):
    """
    Yields (X, y) pairs:
      X: [LOOKBACK_DAYS, n_channels, H, W]  -- input sequence from daily .npy stacks
      y: [n_target_pollutants, H, W]        -- sparse CPCB label grid (NaN at non-station pixels)
    use_val_stations=True builds targets from held-out stations (for the val DataLoader).
    """
    def __init__(self, date_list: list,
                 channel_names: list = None,
                 use_val_stations: bool = False):
        self.dates          = sorted(date_list)
        self.channel_names  = channel_names or INPUT_CHANNEL_NAMES
        self.use_val_stations = use_val_stations
        self.valid_starts   = list(range(len(self.dates) - LOOKBACK_DAYS - PREDICT_HORIZON + 1))

    def __len__(self):
        return len(self.valid_starts)

    def __getitem__(self, idx):
        start       = self.valid_starts[idx]
        seq_dates   = self.dates[start: start + LOOKBACK_DAYS]
        target_date = self.dates[start + LOOKBACK_DAYS + PREDICT_HORIZON - 1]

        seq = []
        for d in seq_dates:
            stack = load_daily_stack(d)
            seq.append(np.stack([stack[ch] for ch in self.channel_names], axis=0))
        X = np.stack(seq, axis=0).astype("float32")  # [T, C, H, W]

        y = populate_target_grid(target_date, CONVLSTM_POLLUTANTS,
                                  use_val_stations=self.use_val_stations)
        return torch.tensor(X), torch.tensor(y)


## 9. Model Architecture

### ConvLSTM branch (PM2.5, PM10, CO) with spatio-temporal attention
### XGBoost stub (SO2, NO2) — replace once real model is wired in
### O3 sub-model — NO2-only (HCHO removed)
### BMA dynamic weighting hook — structural only until XGBoost stub is real
### Optional: Neural-operator alternate core (CoNOAir inspired)


In [ ]:
# ── Spatio-temporal attention (KSC-ConvLSTM inspired) ──────────────────────
class SpatioTemporalAttention(nn.Module):
    def __init__(self, channels: int, hidden_dim: int = 32):
        super().__init__()
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(channels, hidden_dim, 1), nn.ReLU(),
            nn.Conv2d(hidden_dim, 1, 1), nn.Sigmoid())
        self.temporal_gate = nn.Sequential(
            nn.Linear(channels, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1))

    def forward(self, x):
        B, T, C, H, W = x.shape
        sw = torch.stack([self.spatial_gate(x[:, t]) for t in range(T)], dim=1)
        x_s = x * sw
        tl  = self.temporal_gate(x.mean(dim=[3,4])).squeeze(-1)   # [B, T]
        tw  = F.softmax(tl, dim=1).view(B, T, 1, 1, 1)
        return x_s * tw, sw, tw


# ── ConvLSTM cell ──────────────────────────────────────────────────────────
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim,
                               kernel_size, padding=kernel_size//2)

    def forward(self, x, h, c):
        gates   = self.conv(torch.cat([x, h], 1))
        i, f, o, g = torch.chunk(gates, 4, 1)
        c_next  = torch.sigmoid(f)*c + torch.sigmoid(i)*torch.tanh(g)
        h_next  = torch.sigmoid(o)*torch.tanh(c_next)
        return h_next, c_next


# ── ConvLSTM branch ────────────────────────────────────────────────────────
class ConvLSTMBranch(nn.Module):
    def __init__(self, input_channels: int, hidden_dim: int = 64, n_targets: int = 3):
        super().__init__()
        self.attention  = SpatioTemporalAttention(input_channels)
        self.cell       = ConvLSTMCell(input_channels, hidden_dim)
        self.hidden_dim = hidden_dim
        self.head       = nn.Conv2d(hidden_dim, n_targets, 1)

    def forward(self, x):
        x_a, sw, tw = self.attention(x)
        B, T, C, H, W = x_a.shape
        h = torch.zeros(B, self.hidden_dim, H, W, device=x.device)
        c = torch.zeros(B, self.hidden_dim, H, W, device=x.device)
        for t in range(T):
            h, c = self.cell(x_a[:, t], h, c)
        return self.head(h), {"spatial_attention": sw, "temporal_attention": tw}


# ── XGBoost stub ───────────────────────────────────────────────────────────
class XGBoostBranchStub:
    """STUB: replace with real CNN+XGBoost model once available."""
    def predict(self, date_str: str) -> dict:
        raise NotImplementedError("XGBoost stub -- wire in real model.")

xgboost_branch = XGBoostBranchStub()


# ── O3 sub-model ───────────────────────────────────────────────────────────
class O3SubModel(nn.Module):
    """NO2-only input (HCHO removed). Loses NOx-limited vs VOC-limited distinction."""
    def __init__(self, hidden_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(len(O3_INPUT_POLLUTANTS), hidden_dim, 3, padding=1),
            nn.ReLU(), nn.Conv2d(hidden_dim, 1, 1))
    def forward(self, no2_grid): return self.net(no2_grid)


# ── BMA dynamic weighting (structural hook) ────────────────────────────────
class BMADynamicWeighting(nn.Module):
    """STRUCTURAL HOOK ONLY -- cannot train until XGBoost stub is replaced."""
    def __init__(self, context_dim: int = 13):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(context_dim,16), nn.ReLU(), nn.Linear(16,2))
    def forward(self, ctx, conv_pred, xgb_pred):
        w = F.softmax(self.net(ctx), dim=-1)
        return w[...,0].view(-1,1,1,1)*conv_pred + w[...,1].view(-1,1,1,1)*xgb_pred, w


# ── Optional neural-operator core (CoNOAir inspired, benchmarking only) ───
class FractionalFourierOperatorBlock(nn.Module):
    """Simplified FNO-style block. Swap with ConvLSTMBranch for benchmarking."""
    def __init__(self, input_channels, hidden_dim=64, n_targets=3):
        super().__init__()
        self.proj  = nn.Conv2d(input_channels, hidden_dim, 1)
        self.sw    = nn.Parameter(torch.randn(hidden_dim, hidden_dim, dtype=torch.cfloat)*0.02)
        self.out   = nn.Conv2d(hidden_dim, n_targets, 1)
    def forward(self, x):
        h = self.proj(x.mean(1))
        h = torch.fft.ifft2(torch.einsum("bchw,co->bohw",
                             torch.fft.fft2(h,dim=(-2,-1)), self.sw),
                             dim=(-2,-1)).real
        return self.out(h), {}


## 10. Physics-Informed Loss Function

Total loss = MAE (masked to station pixels) + λ·Pearson (spatial pattern) +
λ·PM2.5≤PM10 penalty + λ·advection-diffusion PDE residual.

R² is a **reported metric only** — not a loss term (it's mathematically redundant with MAE/MSE).


In [ ]:
def masked_mae_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    mask = ~torch.isnan(target)
    if mask.sum() == 0:
        return torch.tensor(0.0, device=pred.device, requires_grad=True)
    return torch.abs(pred[mask] - target[mask]).mean()


def pearson_correlation_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """1 - Pearson r at valid station pixels. Penalises wrong spatial pattern."""
    mask = ~torch.isnan(target)
    if mask.sum() < 2:
        return torch.tensor(0.0, device=pred.device, requires_grad=True)
    p, t = pred[mask], target[mask]
    pc, tc = p - p.mean(), t - t.mean()
    r = (pc*tc).sum() / (torch.sqrt((pc**2).sum()*(tc**2).sum()) + 1e-8)
    return 1.0 - r


def pm_ordering_penalty(pm25: torch.Tensor, pm10: torch.Tensor) -> torch.Tensor:
    """Soft constraint: PM2.5 ≤ PM10 (PM2.5 is a subset of PM10 by particle size)."""
    return (F.relu(pm25 - pm10)**2).mean()


def advection_diffusion_residual(pred_seq: torch.Tensor,
                                  u: torch.Tensor, v: torch.Tensor,
                                  grid_spacing_m: float = 5000.0,
                                  dt: float = 86400.0) -> torch.Tensor:
    """
    PDE residual: dC/dt + u·∇C - K·∇²C - S ≈ 0
    pred_seq: [B, 2, H, W] -- two consecutive predicted grids for one pollutant.
    Simplifications: K = DIFFUSIVITY_K (constant), S = 0 (no emissions inventory).
    Grid edges excluded (torch.roll wraps, which is physically wrong at bbox boundary).
    """
    C0, C1 = pred_seq[:, 0], pred_seq[:, 1]
    dC_dt  = (C1 - C0) / dt
    dx     = (torch.roll(C0,-1,-1) - torch.roll(C0,1,-1)) / (2*grid_spacing_m)
    dy     = (torch.roll(C0,-1,-2) - torch.roll(C0,1,-2)) / (2*grid_spacing_m)
    lap    = (torch.roll(C0,-1,-1)+torch.roll(C0,1,-1)+
              torch.roll(C0,-1,-2)+torch.roll(C0,1,-2) - 4*C0) / grid_spacing_m**2
    residual = (dC_dt + u*dx + v*dy - DIFFUSIVITY_K*lap - EMISSION_SOURCE_S)[:, 2:-2, 2:-2]
    return (residual**2).mean()


def compute_total_loss(pred: torch.Tensor, target: torch.Tensor,
                        pred_seq_pde: torch.Tensor = None,
                        u: torch.Tensor = None, v: torch.Tensor = None) -> dict:
    """
    Returns dict of individual loss components + 'total' for backward().
    pred/target: [B, 3, H, W] -- channels ordered per CONVLSTM_POLLUTANTS.
    """
    losses = {}
    losses["mae"]         = masked_mae_loss(pred, target)
    losses["pearson"]     = pearson_correlation_loss(pred, target)

    p25 = CONVLSTM_POLLUTANTS.index("PM2.5")
    p10 = CONVLSTM_POLLUTANTS.index("PM10")
    losses["pm_ordering"] = pm_ordering_penalty(pred[:, p25], pred[:, p10])

    if pred_seq_pde is not None and u is not None and v is not None:
        losses["pde"] = advection_diffusion_residual(pred_seq_pde, u, v)
    else:
        losses["pde"] = torch.tensor(0.0, device=pred.device)

    losses["total"] = (losses["mae"]
                       + LAMBDA_PEARSON      * losses["pearson"]
                       + LAMBDA_PM_ORDERING  * losses["pm_ordering"]
                       + LAMBDA_PDE_RESIDUAL * losses["pde"])
    return losses


## 11. Training Infrastructure (multi-seed ensemble, schedulers, Optuna)

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def build_scheduler(opt, stype="plateau", **kw):
    if stype == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=kw.get("patience", 5))
    return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=kw.get("t_max", 50))


def train_one_seed(seed, train_ds, val_ds,
                    n_epochs=None, lr=1e-3, hidden_dim=64,
                    scheduler_type="plateau", batch_size=None,
                    use_neural_operator=False) -> dict:
    n_epochs   = n_epochs   or FAST_N_EPOCHS
    batch_size = batch_size or FAST_BATCH_SIZE
    set_seed(seed)

    n_ch = len(INPUT_CHANNEL_NAMES)
    model = (FractionalFourierOperatorBlock(n_ch, hidden_dim, len(CONVLSTM_POLLUTANTS))
             if use_neural_operator
             else ConvLSTMBranch(n_ch, hidden_dim, len(CONVLSTM_POLLUTANTS))).to(DEVICE)

    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = build_scheduler(opt, scheduler_type)
    tl    = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    vl    = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)

    history = {"train_loss": [], "val_loss": []}

    # Channel indices for PDE wind extraction (u10, v10 in INPUT_CHANNEL_NAMES)
    u10_idx = INPUT_CHANNEL_NAMES.index("u10") if "u10" in INPUT_CHANNEL_NAMES else None
    v10_idx = INPUT_CHANNEL_NAMES.index("v10") if "v10" in INPUT_CHANNEL_NAMES else None
    p25_idx = CONVLSTM_POLLUTANTS.index("PM2.5")

    for epoch in range(n_epochs):
        model.train()
        ep_loss = []
        for X, y in tl:
            X, y = X.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            pred, _ = model(X)

            # PDE: extract u10/v10 from last timestep; use PM2.5 pred as stand-in for t,t+1 pair
            u = X[:, -1, u10_idx] if u10_idx is not None else None
            v = X[:, -1, v10_idx] if v10_idx is not None else None
            pred_seq = torch.stack([pred[:, p25_idx], pred[:, p25_idx]], dim=1) if u is not None else None

            ls = compute_total_loss(pred, y, pred_seq, u, v)
            ls["total"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss.append(ls["total"].item())

        tl_mean = float(np.mean(ep_loss))
        history["train_loss"].append(tl_mean)

        model.eval()
        vl_loss = []
        with torch.no_grad():
            for X, y in vl:
                X, y = X.to(DEVICE), y.to(DEVICE)
                pred, _ = model(X)
                vl_loss.append(compute_total_loss(pred, y)["total"].item())
        vl_mean = float(np.mean(vl_loss)) if vl_loss else float("nan")
        history["val_loss"].append(vl_mean)

        if scheduler_type == "plateau":
            sched.step(vl_mean)
        else:
            sched.step()

        if epoch % 5 == 0:
            print(f"[seed {seed}] epoch {epoch:3d}: train={tl_mean:.4f}  val={vl_mean:.4f}")

    return {"model": model, "history": history, "seed": seed}


def train_ensemble(train_ds, val_ds, **kw) -> list:
    ensemble = []
    for seed in FAST_GLOBAL_SEEDS:
        print(f"\n=== Training seed {seed} ===")
        ensemble.append(train_one_seed(seed, train_ds, val_ds, **kw))
    return ensemble


def ensemble_predict(ensemble: list, X: torch.Tensor) -> tuple:
    """Returns (mean_pred, std_pred) across seeds. std feeds the uncertainty map."""
    preds = []
    with torch.no_grad():
        for m in ensemble:
            m["model"].eval()
            p, _ = m["model"](X.to(DEVICE))
            preds.append(p.cpu().numpy())
    preds = np.stack(preds, 0)
    return preds.mean(0), preds.std(0)


## 12. Hyperparameter Tuning (Optuna)

In [ ]:
def optuna_objective(trial, train_ds, val_ds):
    global LAMBDA_PM_ORDERING, LAMBDA_PDE_RESIDUAL, LAMBDA_PEARSON
    lr         = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 128])
    stype      = trial.suggest_categorical("scheduler_type", ["plateau", "cosine"])
    LAMBDA_PM_ORDERING  = trial.suggest_float("lambda_pm", 0.01, 0.2)
    LAMBDA_PDE_RESIDUAL = trial.suggest_float("lambda_pde", 0.005, 0.1)
    LAMBDA_PEARSON      = trial.suggest_float("lambda_pearson", 0.01, 0.3)
    res = train_one_seed(FAST_GLOBAL_SEEDS[0], train_ds, val_ds,
                          n_epochs=FAST_OPTUNA_EPOCHS, lr=lr,
                          hidden_dim=hidden_dim, scheduler_type=stype)
    return res["history"]["val_loss"][-1]


def run_hyperparameter_search(train_ds, val_ds, n_trials=None):
    if optuna is None:
        raise ImportError("pip install optuna")
    n_trials = n_trials or FAST_OPTUNA_TRIALS
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda t: optuna_objective(t, train_ds, val_ds), n_trials=n_trials)
    print("Best params:", study.best_params)
    print("Best val loss:", study.best_value)
    return study


## 13. Validation — Leave-One-Station-Out, Stratified by Season & Severity

Metrics (R², MAE, Pearson r) reported **per season and AQI severity bin**, not averaged.
Separate reporting for train-adjacent vs. truly held-out stations.


In [ ]:
AQI_SEVERITY_BINS = {
    "Good": (0,50), "Satisfactory": (50,100), "Moderate": (100,200),
    "Poor": (200,300), "Very Poor": (300,400), "Severe": (400,1000),
}
SEASONS = {
    "Winter": [12,1,2], "Summer": [3,4,5],
    "Monsoon": [6,7,8,9], "Post-Monsoon": [10,11],
}


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mask = ~np.isnan(y_true)
    if mask.sum() < 2:
        return {"r2": np.nan, "mae": np.nan, "pearson_r": np.nan, "n": int(mask.sum())}
    yt, yp = y_true[mask], y_pred[mask]
    mae    = float(np.abs(yt-yp).mean())
    ss_res = float(((yt-yp)**2).sum())
    ss_tot = float(((yt-yt.mean())**2).sum())
    r2     = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    pearson_r = float(np.corrcoef(yt, yp)[0,1]) if len(yt) > 1 else np.nan
    return {"r2": r2, "mae": mae, "pearson_r": pearson_r, "n": int(mask.sum())}


def select_stratified_loso_subset(stations_df: pd.DataFrame, n: int = 12) -> pd.DataFrame:
    if n is None or n >= len(stations_df): return stations_df
    quota  = max(1, n // stations_df["stateID"].nunique())
    picked = stations_df.groupby("stateID", group_keys=False).apply(
                lambda g: g.sample(min(len(g), quota), random_state=42))
    if len(picked) > n:
        picked = picked.sample(n, random_state=42)
    elif len(picked) < n:
        extra = stations_df.drop(picked.index).sample(
                    min(len(stations_df)-len(picked), n-len(picked)), random_state=42)
        picked = pd.concat([picked, extra])
    return picked.reset_index(drop=True)


def loso_validation(ensemble: list, all_dates: list,
                     all_stations_df: pd.DataFrame, pollutant: str = "PM2.5") -> pd.DataFrame:
    """
    Stratified LOSO: loops held-out stations, predicts at each station's pixel
    for all dates, collects results for season×severity breakdown.
    NOTE: full LOSO = N separate training runs. With FAST_CONFIG this runs on
    a 12-station stratified subset only. The loop below infers from the
    already-trained ensemble (no re-training per fold) -- a lighter approximate
    LOSO that checks spatial generalization without full re-training cost.
    """
    results = []
    subset  = select_stratified_loso_subset(all_stations_df, FAST_LOSO_N_STATIONS)

    for _, stn in subset.iterrows():
        try:
            overpass = load_cpcb_overpass(stn["station_name"], CPCB_OVERPASS_VAL_DIR)
        except FileNotFoundError:
            continue
        r, c = latlon_to_pixel(stn["latitude"], stn["longitude"])

        for d in all_dates:
            try:
                stack = load_daily_stack(d)
            except FileNotFoundError:
                continue
            vals = extract_overpass_hour_target(overpass, d)
            if pollutant not in vals or pd.isna(vals.get(pollutant)):
                continue

            month  = pd.Timestamp(d).month
            season = next((s for s,ms in SEASONS.items() if month in ms), "Unknown")
            y_true = float(vals[pollutant])

            # Use ensemble mean prediction at this pixel
            seq_start = all_dates.index(d) - LOOKBACK_DAYS
            if seq_start < 0: continue
            seq_dates = all_dates[seq_start: all_dates.index(d)]
            try:
                seq = np.stack([np.stack([load_daily_stack(sd)[ch]
                                          for ch in INPUT_CHANNEL_NAMES], 0)
                                 for sd in seq_dates], 0)
            except FileNotFoundError:
                continue
            X   = torch.tensor(seq[np.newaxis], dtype=torch.float32)
            mn, _ = ensemble_predict(ensemble, X)
            pol_idx = CONVLSTM_POLLUTANTS.index(pollutant) if pollutant in CONVLSTM_POLLUTANTS else 0
            y_pred  = float(mn[0, pol_idx, r, c])

            sev = next((k for k,(lo,hi) in AQI_SEVERITY_BINS.items() if lo<=y_true<hi), "Severe")
            results.append({"station": stn["station_name"], "date": d,
                             "season": season, "severity_bin": sev,
                             "y_true": y_true, "y_pred": y_pred})

    return pd.DataFrame(results)


def stratified_metrics_report(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for season in SEASONS:
        for sev in AQI_SEVERITY_BINS:
            sub = df[(df["season"]==season) & (df["severity_bin"]==sev)]
            if len(sub) == 0: continue
            m = compute_metrics(sub["y_true"].values, sub["y_pred"].values)
            rows.append({"season": season, "severity_bin": sev, **m})
    return pd.DataFrame(rows)


## 14. CNN + XGBoost Branch (SO2, NO2) — Lag-Free

All lag features removed — works at any grid pixel, no station history needed.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers as klayers, Model as KModel
from tensorflow.keras.callbacks import EarlyStopping

XGB_PATCH_SIZE = 13
XGB_TAB_NAMES  = ["lat","lon","elevation","doy","doy_sin","doy_cos",
                   "ventilation_index","wind_direction","t2m_squared",
                   "solar_radiation","BLH_center","wind_speed_center",
                   "AOD_center","AOD_center_ratio","RH_center"]


def extract_patch(grid: np.ndarray, row: int, col: int,
                   patch_size: int = XGB_PATCH_SIZE) -> np.ndarray:
    half = patch_size // 2
    padded = np.pad(grid, half, mode="edge")
    r, c = row + half, col + half
    return padded[r-half: r+half+1, c-half: c+half+1]


def xgb_input_channels(stack: dict) -> list:
    wanted = ["aod","no2","so2","t2m","u10","v10","blh","sp","tcc","rh_surface","lat_norm","lon_norm"]
    return [ch for ch in wanted if ch in stack]


def build_patch_cnn(input_shape: tuple):
    inp = klayers.Input(shape=input_shape)
    x   = klayers.Conv2D(16, 3, padding="same")(inp)
    x   = klayers.BatchNormalization()(x); x = klayers.LeakyReLU()(x)
    x   = klayers.MaxPool2D(2)(x)
    x   = klayers.Conv2D(32, 3, padding="same")(x)
    x   = klayers.BatchNormalization()(x); x = klayers.LeakyReLU()(x)
    gap = klayers.GlobalAveragePooling2D(name="gap")(x)
    out = klayers.Dense(1)(klayers.Dropout(0.3)(gap))
    return KModel(inp, out), KModel(inp, gap)


def _make_tab_row(row, stack, r, c, doy):
    def get(ch): return stack.get(ch, np.zeros((GRID_HEIGHT,GRID_WIDTH)))[r,c]
    return [
        row["latitude"], row["longitude"], 0.0,
        doy, np.sin(2*np.pi*doy/365), np.cos(2*np.pi*doy/365),
        get("blh"), np.arctan2(get("v10"), get("u10")), get("t2m")**2,
        get("ssrd") if "ssrd" in stack else 0.0,
        get("blh"), np.sqrt(get("u10")**2 + get("v10")**2),
        get("aod"), 1.0, get("rh_surface"),
    ]


def train_cnn_xgb_branch(pollutant: str, stations_df: pd.DataFrame,
                           date_list: list, n_epochs: int = None) -> dict:
    import xgboost as xgb
    n_epochs = n_epochs or FAST_N_EPOCHS
    lookup   = build_station_pixel_lookup(stations_df)
    X_pat, y_tgt, X_tab = [], [], []

    for _, row in stations_df.iterrows():
        try: overpass = load_cpcb_overpass(row["station_name"])
        except FileNotFoundError: continue
        r, c = lookup[row["station_name"]]
        for d in date_list:
            try: stack = load_daily_stack(d)
            except FileNotFoundError: continue
            vals = extract_overpass_hour_target(overpass, d)
            if pollutant not in vals or pd.isna(vals.get(pollutant)): continue
            chs = xgb_input_channels(stack)
            X_pat.append(np.stack([extract_patch(stack[ch],r,c) for ch in chs], -1))
            y_tgt.append(float(vals[pollutant]))
            X_tab.append(_make_tab_row(row, stack, r, c, pd.Timestamp(d).dayofyear))

    if not X_pat:
        raise RuntimeError(f"No samples for {pollutant} -- check data paths.")

    X_pat  = np.array(X_pat,  "float32")
    y_tgt  = np.array(y_tgt,  "float32")
    X_tab  = np.nan_to_num(np.array(X_tab, "float32"))
    idx    = np.random.RandomState(42).permutation(len(X_pat))
    sp     = int(0.8*len(idx))
    tr, vl = idx[:sp], idx[sp:]

    cnn, gap_m = build_patch_cnn(X_pat.shape[1:])
    cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0), loss="huber")
    cnn.fit(X_pat[tr], y_tgt[tr], validation_data=(X_pat[vl],y_tgt[vl]),
            epochs=n_epochs, batch_size=FAST_BATCH_SIZE,
            callbacks=[EarlyStopping("val_loss",patience=5,restore_best_weights=True)], verbose=0)

    cnn_pred = cnn.predict(X_pat, 32, verbose=0).flatten()
    gap_feat = gap_m.predict(X_pat, 32, verbose=0)
    X_xgb    = np.hstack([gap_feat, cnn_pred.reshape(-1,1), X_tab])
    resid    = y_tgt - cnn_pred

    xgb_m = xgb.XGBRegressor(objective="reg:squarederror", tree_method="hist",
                               n_estimators=200, max_depth=5, learning_rate=0.05,
                               n_jobs=-1, random_state=42)
    xgb_m.fit(X_xgb[tr], resid[tr])

    final_vl = np.maximum(cnn_pred[vl] + xgb_m.predict(X_xgb[vl]), 0.0)
    m = compute_metrics(y_tgt[vl], final_vl)
    print(f"[{pollutant}] CNN+XGBoost (lag-free) val: {m}")
    return {"cnn": cnn, "gap": gap_m, "xgb": xgb_m,
            "channels": xgb_input_channels(load_daily_stack(date_list[0])), "metrics": m}


class CNNXGBoostBranch:
    def __init__(self, models: dict): self.models = models

    def predict_grid(self, date_str: str) -> dict:
        """Returns {pollutant: (H,W) array} for SO2 and NO2 across the full grid.
        KNOWN: XGBoost residual step omitted at inference for brevity --
        see train_cnn_xgb_branch for exact feature assembly to complete this."""
        stack = load_daily_stack(date_str)
        out   = {}
        for pol, bun in self.models.items():
            grid = np.full((GRID_HEIGHT,GRID_WIDTH), np.nan, "float32")
            step = DOWNSAMPLE_FACTOR if FAST_CONFIG else 1
            for r in range(0, GRID_HEIGHT, step):
                for c in range(0, GRID_WIDTH, step):
                    p = np.stack([extract_patch(stack[ch],r,c) for ch in bun["channels"]],-1)
                    grid[r,c] = bun["cnn"].predict(p[np.newaxis], verbose=0).flat[0]
            out[pol] = grid
        return out


## 15. Output Generation — AQI Map + Uncertainty Map

In [ ]:
def grid_to_cpcb_aqi(pollutant_grids: dict) -> np.ndarray:
    """Pixel-wise CPCB AQI = max of sub-indices.
    PLACEHOLDER: replace sub-index list with real CPCB breakpoint table."""
    return np.nanmax(np.stack(list(pollutant_grids.values()), 0), 0)


def build_uncertainty_map(ens_std: np.ndarray,
                           pde_res_map: np.ndarray = None,
                           alpha: float = 0.7) -> np.ndarray:
    """Combines ensemble spread + PDE residual magnitude into a [0,1] uncertainty map."""
    def norm(a): return (a - a.min()) / (a.max() - a.min() + 1e-8)
    if pde_res_map is not None:
        return alpha*norm(ens_std) + (1-alpha)*norm(pde_res_map)
    return norm(ens_std)


def plot_aqi_and_uncertainty(aqi: np.ndarray, unc: np.ndarray, date_str: str):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    c0 = axes[0].imshow(aqi, cmap="RdYlGn_r", vmin=0, vmax=500)
    axes[0].set_title(f"Predicted AQI — {date_str}")
    plt.colorbar(c0, ax=axes[0], label="AQI")
    c1 = axes[1].imshow(unc, cmap="viridis", vmin=0, vmax=1)
    axes[1].set_title(f"Uncertainty — {date_str}")
    plt.colorbar(c1, ax=axes[1], label="Normalised uncertainty")
    plt.tight_layout(); plt.show()


## 16. Known Limitations

1. **AOD monsoon gaps** (Jun–Sep ~88% NaN): masked as NaN, not interpolated.
2. **Pressure-level RH unavailable**: ERA5 pressure-level pull has only temperature `t`.
   Lapse-rate diagnostic is temperature-only — not full moisture-aware stability.
3. **PDE simplifications**: K (diffusivity) is a constant; S (emissions) = 0 (no inventory).
4. **2020 COVID anomaly**: flagged via `lockdown_flag` channel, not excluded.
5. **O3 sub-model**: NO2-only input (HCHO removed). Loses NOx-limited vs VOC-limited distinction.
6. **XGBoost stub + BMA layer**: structural hooks only — cannot train until real XGBoost outputs wired in.
7. **LOSO in FAST_CONFIG**: 12-station approximate LOSO using the already-trained ensemble
   (no per-fold retraining). Full LOSO (N re-trains) is compute-infeasible on a single T4.
8. **`predict_grid` XGBoost residual step**: incomplete — CNN-only at inference.
   Full inference requires the tabular feature assembly from `train_cnn_xgb_branch` at each pixel.
9. **Notebook cannot run fully end-to-end until** all Stage-2 `.npy` files exist in Drive
   at `FINAL_INPUT_DIR` and all CPCB overpass CSVs exist in `CPCB_OVERPASS_DIR`/`val/`.


## 17. Execution Block

**Run this cell last.** All definitions above must be executed first (Sections 0–16).
This cell: builds datasets → Optuna search → trains ConvLSTM ensemble → trains CNN+XGBoost →
runs stratified LOSO → prints final metrics.


In [ ]:
# ── 1. Date splits (temporal, no shuffling) ──────────────────────────────────
train_dates = pd.date_range(START_DATE, "2022-12-31").strftime("%Y-%m-%d").tolist()
val_dates   = pd.date_range("2023-01-01", END_DATE).strftime("%Y-%m-%d").tolist()
print(f"Train dates: {len(train_dates)} | Val dates: {len(val_dates)}")

# ── 2. Build sequence datasets ───────────────────────────────────────────────
train_dataset = AQISequenceDataset(train_dates, INPUT_CHANNEL_NAMES, use_val_stations=False)
val_dataset   = AQISequenceDataset(val_dates,   INPUT_CHANNEL_NAMES, use_val_stations=True)
print(f"Train sequences: {len(train_dataset)} | Val sequences: {len(val_dataset)}")

# ── 3. Optuna hyperparameter search ─────────────────────────────────────────
print("\n--- Optuna search ---")
study = run_hyperparameter_search(train_dataset, val_dataset)
best  = study.best_params
print("Best params:", best)

# ── 4. Train ConvLSTM ensemble with best params ──────────────────────────────
print("\n--- Training ConvLSTM ensemble (PM2.5, PM10, CO) ---")
ensemble = train_ensemble(
    train_dataset, val_dataset,
    n_epochs       = FAST_N_EPOCHS,
    lr             = best.get("lr", 1e-3),
    hidden_dim     = best.get("hidden_dim", 64),
    scheduler_type = best.get("scheduler_type", "plateau"),
    batch_size     = FAST_BATCH_SIZE,
)

# ── 5. Train CNN+XGBoost branch (SO2, NO2) ──────────────────────────────────
print("\n--- Training CNN+XGBoost branch (SO2, NO2) ---")
so2_bundle = train_cnn_xgb_branch("SO2", train_stations_df, train_dates)
no2_bundle = train_cnn_xgb_branch("NO2", train_stations_df, train_dates)
cnn_xgb    = CNNXGBoostBranch({"SO2": so2_bundle, "NO2": no2_bundle})

# ── 6. Stratified LOSO validation ────────────────────────────────────────────
print("\n--- Stratified LOSO validation ---")
loso_df  = loso_validation(ensemble, val_dates, val_stations_df, pollutant="PM2.5")
report   = stratified_metrics_report(loso_df)
print("\nLOSO metrics (PM2.5, season × severity):")
print(report.to_string(index=False))

# ── 7. Sample AQI + uncertainty map for last val date ────────────────────────
print("\n--- Generating sample AQI map ---")
sample_date = val_dates[-1]
sample_stack = load_daily_stack(sample_date)
# Build a single-sample batch from last LOOKBACK_DAYS of val_dates
seq = np.stack([np.stack([load_daily_stack(d)[ch] for ch in INPUT_CHANNEL_NAMES],0)
                 for d in val_dates[-LOOKBACK_DAYS:]], 0)
X_sample  = torch.tensor(seq[np.newaxis], dtype=torch.float32)
mean_pred, std_pred = ensemble_predict(ensemble, X_sample)

p25i, p10i = CONVLSTM_POLLUTANTS.index("PM2.5"), CONVLSTM_POLLUTANTS.index("PM10")
aqi_map = grid_to_cpcb_aqi({
    "PM2.5": mean_pred[0, p25i],
    "PM10":  mean_pred[0, p10i],
})
unc_map = build_uncertainty_map(std_pred[0].mean(0))
plot_aqi_and_uncertainty(aqi_map, unc_map, sample_date)

print("\n✅ Pipeline complete.")
